<a href="https://colab.research.google.com/github/we-human-centric/CursoPythonDatos_2026/blob/main/Dia%2010%20-%202026-05-06%20-%20Regresi%C3%B3n%20Log%C3%ADstica/clase_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Día 10 · Regresión Logística + SMOTE

**Seminario EDA 2026 — Semana 3, Día 3 (miércoles 6 de mayo)**

> Pasamos de predecir números (regresión) a predecir **categorías** (clasificación binaria). Aprendemos:
>
> 1. Por qué una recta no sirve y cómo la sigmoide la reemplaza.
> 2. Cómo se interpretan los coeficientes (log-odds → odds ratio).
> 3. El problema de las clases desbalanceadas y cómo lo aliviamos con **SMOTE**.
> 4. Regularización L1 / L2 y el parámetro `C`.

Acompaña la clase con: 🎮 [confusion-matrix](../simuladores/simulador-confusion-matrix.html) · [overfitting](../simuladores/simulador-overfitting.html)

## 0. Setup

Si estás en Colab y `imbalanced-learn` no está, descomenta la primera línea.

In [ ]:
# !pip install -q imbalanced-learn scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, f1_score, precision_score, recall_score,
                             roc_curve)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

SEED = 42
np.random.seed(SEED)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
print('Setup OK')

## 1. Dataset Pima Indians Diabetes

768 mujeres, 8 features clínicas, 1 variable binaria (`Outcome`: 1 = diabetes, 0 = no diabetes). Es un dataset clásico de clasificación con desbalance moderado (~35% positivos).

In [ ]:
URL = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
cols = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
        'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
df = pd.read_csv(URL, names=cols)
print(df.shape)
df.head()

In [ ]:
# Distribución de la variable objetivo: ¿está balanceada?
balance = df['Outcome'].value_counts(normalize=True).round(3)
print('Distribución Outcome:')
print(balance)
print(f'\nRazón clase 0 / clase 1 = {balance[0]/balance[1]:.2f}')

**Lectura:** ~65% no-diabetes vs ~35% diabetes. Es un desbalance moderado — todavía manejable, pero sí afecta la métrica accuracy. Más adelante veremos qué pasa si lo ignoramos vs si lo corregimos con SMOTE.

## 2. Por qué NO usar regresión lineal aquí

Probemos forzar una `LinearRegression` sobre `Outcome` (0/1) para ver el desastre.

In [ ]:
from sklearn.linear_model import LinearRegression

X1 = df[['Glucose']].values
y = df['Outcome'].values

lr = LinearRegression().fit(X1, y)
xs = np.linspace(X1.min(), X1.max(), 100).reshape(-1, 1)
preds_lin = lr.predict(xs)

# Comparamos con sigmoide (logística)
log = LogisticRegression().fit(X1, y)
preds_log = log.predict_proba(xs)[:, 1]

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(X1, y, alpha=0.3, s=20, label='Datos (0=no, 1=sí)')
ax.plot(xs, preds_lin, 'r--', lw=2, label='Regresión LINEAL (mal)')
ax.plot(xs, preds_log, 'g-',  lw=2, label='Regresión LOGÍSTICA (sigmoide)')
ax.axhline(0, color='gray', lw=0.5); ax.axhline(1, color='gray', lw=0.5)
ax.set_xlabel('Glucose'); ax.set_ylabel('P(diabetes=1)')
ax.set_title('Lineal vs Logística sobre variable binaria')
ax.legend(); plt.tight_layout(); plt.show()

**Lo que ves:**  
La recta roja predice probabilidades < 0 y > 1, sin sentido. La curva sigmoide verde se queda siempre entre 0 y 1, y se **acelera** en la zona donde el cambio de etiqueta es más probable. Por eso usamos logística para clasificar.

## 3. La sigmoide y los log-odds

$$\sigma(z) = \frac{1}{1+e^{-z}} \qquad \text{con} \qquad z = \beta_0 + \beta_1 x_1 + \dots + \beta_p x_p$$

El modelo es lineal **en log-odds**: $\log\dfrac{p}{1-p} = z$. Para interpretar $\beta_j$ exponenciamos: $e^{\beta_j}$ = **odds ratio**.

In [ ]:
# Visualización de la función sigmoide
z = np.linspace(-7, 7, 200)
sig = 1 / (1 + np.exp(-z))
plt.plot(z, sig, lw=2)
plt.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='umbral default = 0.5')
plt.axvline(0,    color='red', linestyle='--', alpha=0.5)
plt.xlabel('z = β₀ + β·x'); plt.ylabel('σ(z) = P(y=1|x)')
plt.title('Función sigmoide')
plt.legend(); plt.tight_layout(); plt.show()

## 4. Modelo logístico completo (sin balanceo todavía)

Buena práctica: escalar las features antes (la regularización es sensible a la escala). Para no fugar info, escalamos dentro del Pipeline.

In [ ]:
X = df.drop(columns='Outcome')
y = df['Outcome'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=1000, random_state=SEED))
])
pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_test)
y_prob = pipe.predict_proba(X_test)[:, 1]

print('Coeficientes (en escala estandarizada):')
for nombre, beta in zip(X.columns, pipe.named_steps['logreg'].coef_[0]):
    print(f'  β_{nombre:<28s} = {beta:+.4f}   →   OR ≈ {np.exp(beta):.3f}')
print(f'\nIntercept = {pipe.named_steps["logreg"].intercept_[0]:+.3f}')

In [ ]:
# Métricas — el avance: por ahora solo accuracy + reporte rápido. Mañana cubrimos métricas en detalle.
print(classification_report(y_test, y_pred, target_names=['No diabetes', 'Diabetes']))
print(f'ROC-AUC (test) = {roc_auc_score(y_test, y_prob):.3f}')

**Mira el reporte con calma:**
- Accuracy global ~77% — parece razonable.
- Pero `recall` para la clase 1 (Diabetes) es solo ~57%: detectamos solo 57% de los casos reales. Eso es grave en medicina (43% de pacientes con diabetes pasan inadvertidos).
- El modelo está sesgado hacia la clase mayoritaria (no-diabetes) porque hay más ejemplos.

Solución: **balancear las clases**.

## 5. SMOTE — Synthetic Minority Over-sampling Technique

**Idea:** en vez de duplicar ejemplos de la clase minoritaria (que sería sobreajuste puro), SMOTE **interpola** entre vecinos cercanos para crear puntos sintéticos. Cada punto nuevo es una combinación convexa de un punto real y uno de sus k-vecinos más cercanos.

**Regla de oro: SMOTE solo se aplica al TRAIN, jamás al TEST.** Aplicarlo al test fugaría datos sintéticos a la evaluación, inflando artificialmente las métricas.

In [ ]:
# Distribución antes y después de SMOTE
sm = SMOTE(random_state=SEED)
X_res, y_res = sm.fit_resample(X_train, y_train)

print('Distribución TRAIN antes de SMOTE:')
print(pd.Series(y_train).value_counts())
print('\nDistribución TRAIN después de SMOTE:')
print(pd.Series(y_res).value_counts())

Ahora repetimos el modelo, **pero envolviendo SMOTE en el Pipeline** (con `imblearn.pipeline.Pipeline` que sí entiende los pasos de resampling). Así, en cada fold de CV, SMOTE se aplica solo al fold de train.

In [ ]:
pipe_smote = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote',  SMOTE(random_state=SEED)),
    ('logreg', LogisticRegression(max_iter=1000, random_state=SEED))
])
pipe_smote.fit(X_train, y_train)

y_pred_s = pipe_smote.predict(X_test)
y_prob_s = pipe_smote.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_s, target_names=['No diabetes', 'Diabetes']))
print(f'ROC-AUC (test) = {roc_auc_score(y_test, y_prob_s):.3f}')

In [ ]:
# Comparativa lado a lado: sin SMOTE vs con SMOTE
comp = pd.DataFrame({
    'Métrica':       ['Accuracy', 'Precision (1)', 'Recall (1)', 'F1 (1)', 'ROC-AUC'],
    'Sin SMOTE':     [(y_pred == y_test).mean(),
                      precision_score(y_test, y_pred),
                      recall_score(y_test, y_pred),
                      f1_score(y_test, y_pred),
                      roc_auc_score(y_test, y_prob)],
    'Con SMOTE':     [(y_pred_s == y_test).mean(),
                      precision_score(y_test, y_pred_s),
                      recall_score(y_test, y_pred_s),
                      f1_score(y_test, y_pred_s),
                      roc_auc_score(y_test, y_prob_s)],
})
comp['Δ'] = comp['Con SMOTE'] - comp['Sin SMOTE']
print(comp.round(3).to_string(index=False))

**Lectura típica:**
- **Recall (clase 1) sube** notablemente — atrapamos más diabetes.
- **Precision (clase 1) puede bajar** un poco — generamos algunos falsos positivos extras.
- **Accuracy global puede bajar** ligeramente — porque ahora cuesta más predecir la clase mayoritaria.
- **F1 generalmente sube** — es la media armónica precision-recall.
- **AUC se mantiene o sube ligeramente** — depende del dataset.

👉 La elección no es "siempre SMOTE". Es: **¿cuál métrica importa para mi negocio?**

## 6. Regularización L1 / L2 y el parámetro `C`

Cuando hay muchas features (o están correlacionadas), los coeficientes pueden inflarse y memorizar ruido. La regularización los penaliza.

- **L2** (Ridge): penaliza la suma de cuadrados de los coeficientes. Los reduce todos a la vez. Bueno cuando hay multicolinealidad.
- **L1** (Lasso): penaliza la suma de valores absolutos. Manda algunos a CERO → selección de features.
- **ElasticNet**: combina L1 y L2 con un mix.

El parámetro `C` en sklearn es el INVERSO de la fuerza de regularización: `C` chico → mucha regularización; `C` grande → poca.

In [ ]:
# Comparamos L2 vs L1 vs sin regularización a varios valores de C
filas = []
for penalty, solver in [('l2', 'lbfgs'), ('l1', 'liblinear')]:
    for C in [0.01, 0.1, 1, 10]:
        m = Pipeline([
            ('s', StandardScaler()),
            ('lr', LogisticRegression(penalty=penalty, C=C, solver=solver,
                                       max_iter=2000, random_state=SEED))
        ]).fit(X_train, y_train)
        coefs = m.named_steps['lr'].coef_[0]
        n_zero = (np.abs(coefs) < 1e-6).sum()
        auc = roc_auc_score(y_test, m.predict_proba(X_test)[:, 1])
        filas.append({'penalty': penalty, 'C': C, 'n_coef_zero': n_zero,
                      'AUC_test': auc, '|β|_max': np.abs(coefs).max()})

tabla_reg = pd.DataFrame(filas).round(4)
print(tabla_reg.to_string(index=False))

**Lo que observas:** con `penalty='l1'` y `C` chico, varios coeficientes se anulan — el modelo hace selección de variables solo. Con L2 los coeficientes se encogen pero no a cero.

## 7. Curva ROC — un primer vistazo

Mañana profundizamos. Aquí solo el plot.

In [ ]:
fpr_a, tpr_a, _ = roc_curve(y_test, y_prob)
fpr_b, tpr_b, _ = roc_curve(y_test, y_prob_s)

plt.figure(figsize=(7, 6))
plt.plot(fpr_a, tpr_a, lw=2, label=f'Sin SMOTE (AUC={roc_auc_score(y_test, y_prob):.3f})')
plt.plot(fpr_b, tpr_b, lw=2, label=f'Con SMOTE (AUC={roc_auc_score(y_test, y_prob_s):.3f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='clasificador aleatorio')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('Curva ROC — Pima Diabetes')
plt.legend(); plt.tight_layout(); plt.show()

## 8. Ejercicio guiado — replica con threshold ajustado

El umbral default 0.5 favorece la clase mayoritaria. Si en tu negocio detectar diabetes es prioritario (FN caros), bájalo. Mira cómo cambia recall.

**TODO:** completa la celda y descubre el umbral que maximiza F1.

In [ ]:
# TODO: completa los _ con un rango de umbrales y elige el mejor F1
thresholds = np.linspace(0.1, 0.9, 17)
f1s = []
for t in thresholds:
    pred_t = (y_prob_s >= t).astype(int)   # ← compara con t
    f1s.append(f1_score(y_test, pred_t))

best_t = thresholds[np.argmax(f1s)]
print(f'Mejor umbral por F1: {best_t:.2f}  →  F1 = {max(f1s):.3f}')

plt.plot(thresholds, f1s, marker='o')
plt.axvline(best_t, color='red', linestyle='--', alpha=0.5)
plt.xlabel('Umbral'); plt.ylabel('F1 (clase 1)')
plt.title('F1 vs umbral de decisión')
plt.tight_layout(); plt.show()

In [ ]:
# Asserts del ejercicio (pasan automáticamente porque la celda anterior está completa)
assert 0 < best_t < 1, 'El umbral debe estar entre 0 y 1'
assert max(f1s) > 0.5, 'F1 esperado > 0.5'
print('✓ Ejercicio completado')

## 9. Validación cruzada — robustez del modelo

Una sola partición train/test puede ser engañosa. Usamos `StratifiedKFold` (mantiene la proporción de clases en cada fold) para estimar AUC con incertidumbre.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scores = cross_val_score(pipe_smote, X, y, scoring='roc_auc', cv=skf)
print(f'AUC por fold: {np.round(scores, 3)}')
print(f'AUC media ± std: {scores.mean():.3f} ± {scores.std():.3f}')

## 10. Entregable parcial S3 (avance)

Sobre el dataset de tu proyecto, identifica una variable objetivo binaria (puede ser una creada con un umbral, p. ej. `compra_alta = ventas > mediana`).

1. Ajusta `LogisticRegression` con `StandardScaler` en pipeline.  
2. Reporta los coeficientes en odds ratios y comenta los 3 más importantes en lenguaje de negocio.  
3. Si la clase positiva está debajo del 35%, aplica SMOTE y compara métricas (precisión, recall, F1, AUC) antes y después.  
4. Prueba `penalty='l1'` con varios `C`. ¿Cuántos coeficientes se anulan?  
5. Encuentra el umbral que maximiza F1 (o el que tu negocio dictaría).

---

**Recordatorio de lo que llevamos a la siguiente clase:**

Mañana (jueves) sustituimos la frontera lineal por una **no lineal**: árboles de decisión. El viernes formalizamos cómo elegir el mejor modelo entre los tres.